 Continuación Limpieza, organización y reconstrucción del conjunto de datos de caracterización estudiantil
**Programa académico**: Ingeneria de Sistemas y Computación<br>
**Sede**: UNIDAD REGIONAL, SEDE FUSAGASUGA <br>
**Elaborado por**: Jesús Villarraga <br>
**Proposito**: El siguiente cuadernillo busca consolidar un cojunto de datos limpio de la informacion socio economica de los estudiantes con el fin de tener variables de poder predictivo sufientemente robusto para predecir la decersión estudiantil. es de resaltar que este cuadernillo es la continuación del cuadernillo **exploración de Datos socio-economicos**; debido a su extensión, por orden y claridad se decidió dividirlo en dos

In [2]:
from operator import index

import pandas as pd
import numpy as np
import re

In [3]:
df_caracterizacion = pd.read_excel("../data/df_caracterizacion_recategorizacion_variables.xlsx")

In [4]:
df_caracterizacion.head()

,DOCUMENTO,GENERO,UBICACION_SEMESTRAL,MEDIO_TRANSPORTE,FRECUENCIA_TRANSPORTE,HORARIO_INICIO_TRANSPORTE,HORARIO_FIN_TRANSPORTE,FUENTES_FINANCIACION,PERSONAS_A_CARGO,DIFICULTAD_PAGO_MATRICULA,...,CONSUME_SUSTANCIAS_PSICOACTIVAS,TIENE_TITULO,OTROS_ESTUDIOS,ORIENTACION_CAT,COMUNA,NIV_EDUCATIVO_PADRE_CAT,NIV_EDUCATIVO_MADRE_CAT,ES_VULNERABLE,ES_GRUPO_ETNICO,GRUPO_ETNICO_CAT
0,1069737604,M,2,Bicicleta,4 veces al dia,06:30:00,18:00:00,Recursos propios,3,Si,...,Nunca,NaN,NaN,Heterosexual,Comuna Sur Occidental,Media,Media,No,No,No étnico
1,1026290752,M,9,Vehículo motorizado,1 vez al dia,12:00:00,22:00:00,Recursos propios,1,Si,...,Nunca,NaN,NaN,Heterosexual,Bogota,Media,Media,No,No,No étnico
2,1069750774,M,9,A pie,1 vez al dia,05:30:00,18:00:00,Recursos propios,1,Si,...,Nunca,NaN,NaN,No responde,Comuna Sur Occidental,Media,Media,No,No,No étnico
3,1069747785,M,5,A pie,mas de 5 veces al dia,06:00:00,21:00:00,Otro / No aplica,0,Si,...,Nunca,NaN,NaN,No responde,Comuna Occidental,Superior,Superior,No,No,No étnico
4,1069760977,M,9,A pie,2 veces al dia,06:00:00,21:00:00,Recursos propios,0,Si,...,Nunca,NaN,NaN,Heterosexual,Comuna Sur Oriental,Básica,Básica,No,No,No étnico


In [5]:
df_caracterizacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830 entries, 0 to 829
Data columns (total 66 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   DOCUMENTO                           830 non-null    int64  
 1   GENERO                              830 non-null    object 
 2   UBICACION_SEMESTRAL                 830 non-null    int64  
 3   MEDIO_TRANSPORTE                    830 non-null    object 
 4   FRECUENCIA_TRANSPORTE               830 non-null    object 
 5   HORARIO_INICIO_TRANSPORTE           830 non-null    object 
 6   HORARIO_FIN_TRANSPORTE              830 non-null    object 
 7   FUENTES_FINANCIACION                830 non-null    object 
 8   PERSONAS_A_CARGO                    830 non-null    int64  
 9   DIFICULTAD_PAGO_MATRICULA           830 non-null    object 
 10  BARRIO_PUNTO_PARTIDA                829 non-null    object 
 11  TURNO_TRABAJO_ESTUDIO               830 non-n

In [6]:
def recategorizar_frecuencia(valor):
    if valor in ['1 vez al dia', '2 veces al dia']:
        return 'Baja frecuencia'
    elif valor in ['3 veces al dia', '4 veces al dia', 'más de 5 veces al dia', 'mas de 5 veces al dia']:
        return 'Alta frecuencia'
    else:
        return 'Otro'  # Por si hay valores inesperados



In [7]:
df_caracterizacion['FRECUENCIA_TRANSPORTE'] = df_caracterizacion['FRECUENCIA_TRANSPORTE'].apply(recategorizar_frecuencia)


In [8]:
df_caracterizacion['FRECUENCIA_TRANSPORTE'].value_counts()

FRECUENCIA_TRANSPORTE
Baja frecuencia    652
Alta frecuencia    121
Otro                57
Name: count, dtype: int64

In [9]:
df_caracterizacion['HORARIO_INICIO_TRANSPORTE'].value_counts()

HORARIO_INICIO_TRANSPORTE
06:30:00    209
06:00:00    185
07:00:00    177
05:00:00     59
05:30:00     52
08:00:00     40
07:30:00     30
09:00:00     16
08:30:00     11
12:00:00      7
10:00:00      6
13:30:00      5
17:00:00      4
16:00:00      4
09:30:00      3
11:00:00      3
11:30:00      3
14:00:00      2
19:00:00      2
15:30:00      2
14:30:00      2
13:00:00      2
19:30:00      1
12:30:00      1
16:30:00      1
18:00:00      1
10:30:00      1
17:30:00      1
Name: count, dtype: int64

In [10]:
df_caracterizacion['HORARIO_FIN_TRANSPORTE'].value_counts()

HORARIO_FIN_TRANSPORTE
18:00:00    132
17:00:00    100
19:00:00     78
20:00:00     72
16:00:00     51
18:30:00     49
21:00:00     48
08:00:00     36
16:30:00     23
19:30:00     23
20:30:00     23
22:00:00     22
15:00:00     22
17:30:00     21
14:00:00     19
13:00:00     18
12:00:00     17
21:30:00     15
22:30:00     11
15:30:00      9
09:00:00      8
12:30:00      8
10:00:00      5
08:30:00      5
09:30:00      4
14:30:00      3
11:30:00      3
11:00:00      2
13:30:00      2
10:30:00      1
Name: count, dtype: int64

In [11]:
ini = pd.to_datetime(df_caracterizacion["HORARIO_INICIO_TRANSPORTE"], format="%H:%M:%S", errors="coerce")
fin = pd.to_datetime(df_caracterizacion["HORARIO_FIN_TRANSPORTE"],   format="%H:%M:%S", errors="coerce")


In [12]:
h_ini = ini.dt.hour + ini.dt.minute/60
h_fin = fin.dt.hour + fin.dt.minute/60

horas = h_fin - h_ini
horas = np.where(horas < 0, horas + 24, horas)  # si cruza medianoche

df_caracterizacion["HORAS_FUERA_CASA"] = np.round(horas, 2)

In [13]:
df_caracterizacion["HORAS_FUERA_CASA"] = pd.qcut(
    df_caracterizacion["HORAS_FUERA_CASA"], q=3, labels=["BAJO", "MEDIO", "ALTO"]
)

In [14]:
df_caracterizacion["HORAS_FUERA_CASA"].value_counts()

HORAS_FUERA_CASA
BAJO     302
ALTO     271
MEDIO    257
Name: count, dtype: int64

A partir de HORARIO_INICIO_TRANSPORTE y HORARIO_FIN_TRANSPORTE se construyó la variable integrada HORAS_FUERA_CASA, discretizada en tres niveles (BAJO, MEDIO, ALTO) con cortes guiados por los datos. Distribución: BAJO = 302, MEDIO = 257, ALTO = 271. Esta síntesis captura la exposición temporal diaria del estudiante y ofrece un indicador más estable que reglas puntuales de inicio/fin.

In [15]:
df_caracterizacion['PERSONAS_A_CARGO'].value_counts()

PERSONAS_A_CARGO
 0             683
 1              64
 2              59
 3              19
 4               2
 31              1
 3196859613      1
-1               1
Name: count, dtype: int64

In [16]:
s = pd.to_numeric(df_caracterizacion["PERSONAS_A_CARGO"], errors="coerce").clip(lower=0)
s = s.where(s <= 10, np.nan)                         # limpia outliers (teléfonos, etc.)

df_caracterizacion["PERSONAS_A_CARGO"] = (s >= 1).astype(int)          # binaria útil

In [17]:
df_caracterizacion["PERSONAS_A_CARGO"] = (
    df_caracterizacion["PERSONAS_A_CARGO"]
    .map({0: "SIN_CARGAS", 1: "CON_CARGAS"})
    .astype("category")
)


In [18]:
df_caracterizacion["PERSONAS_A_CARGO"].value_counts()

PERSONAS_A_CARGO
SIN_CARGAS    686
CON_CARGAS    144
Name: count, dtype: int64

Se transformo la variable "PERSONAS_A_CARGO" en una variable dicotomica partiendo entendiendo que cuando toma el valor CON_CARGA el estudiante reporte al menos 1 persona acargo

In [19]:
df_caracterizacion["DIFICULTAD_PAGO_MATRICULA"].value_counts()

DIFICULTAD_PAGO_MATRICULA
No    507
Si    323
Name: count, dtype: int64

In [20]:
df_caracterizacion['TIPO_VIVIENDA_RESIDENCIA'].value_counts()

TIPO_VIVIENDA_RESIDENCIA
Arrendada              528
Propia pagada          220
Propia pagandose        81
artritis reumatoide      1
Name: count, dtype: int64

In [21]:
df_caracterizacion["TIPO_VIVIENDA_RESIDENCIA"] = (
    df_caracterizacion["TIPO_VIVIENDA_RESIDENCIA"]
    .replace({"artritis reumatoide": "Arrendada"})
)
df_caracterizacion['TIPO_VIVIENDA_RESIDENCIA'].value_counts()

TIPO_VIVIENDA_RESIDENCIA
Arrendada           529
Propia pagada       220
Propia pagandose     81
Name: count, dtype: int64

Se corrige el error de artiritis reumatoide y se le asigna a la mayor frecuencia que en este caso es Arrendada

In [22]:
cols = ["CUENTA_COMPUTADOR", "CUENTA_SMARTHPHONE_TABLET", "CUENTA_INTERNET"]
cnt = (df_caracterizacion[cols].apply(lambda x: x.str.upper().replace({"si":"si"}))
       .replace({"SI":1, "NO":0})
       .astype(int)
       .sum(axis=1))

C:\Users\DILAN\AppData\Local\Temp\ipykernel_6132\3243962140.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"SI":1, "NO":0})


In [23]:
df_caracterizacion["RECURSOS_TECNOLOGICOS"] = pd.cut(
    cnt, bins=[-1, 0, 2, 3], labels=["SIN_RECURSOS", "BASICOS", "COMPLETOS"]
)

In [24]:
df_caracterizacion["RECURSOS_TECNOLOGICOS"].value_counts()

RECURSOS_TECNOLOGICOS
COMPLETOS       520
BASICOS         294
SIN_RECURSOS     16
Name: count, dtype: int64

Se unificaron CUENTA_COMPUTADOR, CUENTA_SMARTHPHONE_TABLET y CUENTA_INTERNET en RECURSOS_TECNOLOGICOS (SIN_RECURSOS / BASICOS / COMPLETOS). Reduce redundancia y resume el acceso digital del estudiante, un factor clave para el rendimiento y el riesgo de deserción

In [25]:
s = pd.to_numeric(df_caracterizacion["NUM_HERMANOS"], errors="coerce").clip(lower=0)
s = s.where(s <= 12, np.nan)  # descarta outliers imposibles (teléfonos, etc.)

df_caracterizacion["HERMANOS_CAT"] = pd.cut(
    s, [-0.1, 0, 2, 12],
    labels=["NINGUNO", "POCOS_1_2", "VARIOS_3_MAS"]
)

In [26]:
df_caracterizacion["HERMANOS_CAT"].value_counts()

HERMANOS_CAT
POCOS_1_2       509
VARIOS_3_MAS    218
NINGUNO         101
Name: count, dtype: int64

Se reemplazó NUM_HERMANOS por la variable cualitativa HERMANOS_CAT (NINGUNO / POCOS_1_2 / VARIOS_3_MAS) porque el campo original contenía errores evidentes (valores imposibles como teléfonos) y, aun limpio, su interpretación numérica es débil. La versión categórica reduce ruido, es más robusta frente a outliers y captura mejor el gradiente sociofamiliar (0, 1–2, 3+) relevante para el riesgo de deserción

In [27]:
df_caracterizacion["HORAS_TRABAJADAS"].describe()

count      816.000000
mean        50.238971
std       1025.120313
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      28812.000000
Name: HORAS_TRABAJADAS, dtype: float64

In [28]:
df_caracterizacion["TRABAJA_ACTUALMENTE"].value_counts()

TRABAJA_ACTUALMENTE
No    681
Si    149
Name: count, dtype: int64

In [29]:
df_caracterizacion["HORAS_TRABAJADAS"].value_counts()

HORAS_TRABAJADAS
0.0        679
8.0         26
10.0        13
16.0        12
40.0        12
20.0        10
30.0         7
24.0         5
42.0         3
9.0          3
26.0         3
5.0          3
48.0         3
12.0         3
2.0          3
56.0         2
2100.0       2
11.0         2
60.0         2
50.0         2
4.0          2
45.0         2
63.0         1
18.0         1
150.0        1
6.0          1
500.0        1
80.0         1
120.0        1
4380.0       1
28.0         1
28812.0      1
22.0         1
14.0         1
47.0         1
70.0         1
39.0         1
15.0         1
7.0          1
Name: count, dtype: int64

In [30]:
trab = (df_caracterizacion["TRABAJA_ACTUALMENTE"]
        .astype(str).str.strip().str.upper()
        .replace({"SI":1, "SÍ":1, "NO":0}).astype(int))

C:\Users\DILAN\AppData\Local\Temp\ipykernel_6132\2960101828.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"SI":1, "SÍ":1, "NO":0}).astype(int))


In [31]:
hrs = pd.to_numeric(df_caracterizacion["HORAS_TRABAJADAS"], errors="coerce").clip(lower=0)
hrs = hrs.mask(trab == 0, 0)          # si NO trabaja => 0
hrs = hrs.mask(hrs > 84, np.nan)      # outliers (>84 h/sem) => NaN
med_trab = hrs[trab == 1].median()    # mediana entre quienes sí trabajan
hrs.loc[(trab == 1) & (hrs.isna())] = med_trab  # imputa solo a quienes trabajan


In [32]:
df_caracterizacion["CARGA_LABORAL"] = pd.cut(
    hrs, bins=[-0.1, 0, 20, 84],
    labels=["NO_TRABAJA", "PARCIAL_1_20", "ALTA_>20"]
).astype("category")

Se fusionaron TRABAJA_ACTUALMENTE y HORAS_TRABAJADAS en una sola variable categórica CARGA_LABORAL con tres niveles: NO_TRABAJA, PARCIAL_1_20 y ALTA_>20. Resume la carga de tiempo laboral (fricción con el estudio) en un indicador interpretable y listo para one-hot encoding en el modelo de deserción.

In [33]:
pv = pd.to_numeric(df_caracterizacion["PERSONAS_VIVE"], errors="coerce").clip(lower=0)
pv = pv.where(pv <= 15, np.nan)   # ej. 656564, 50 -> NaN

nn = pd.to_numeric(df_caracterizacion["NUM_NIÑOS_VIVE"], errors="coerce").clip(lower=0)
nn = nn.where(nn <= 8, np.nan)    # ej. -1 -> 0; 9+ -> NaN


In [34]:
cats = np.select(
    [
        (pv <= 1) & (nn == 0),     # vive solo, sin niños
        (pv >= 2) & (nn == 0),     # hogar sin niños
        nn.between(1, 2, inclusive="both"),  # 1–2 niños
        (nn >= 3)                  # 3+ niños
    ],
    ["SOLO_SIN_NINOS", "HOGAR_SIN_NINOS", "CON_NINOS_1_2", "CON_NINOS_3_MAS"],
    default="DESCONOCIDO"
)

In [35]:
df_caracterizacion["CARGA_HOGAR"] = pd.Series(cats, index=df_caracterizacion.index).astype("category")

In [36]:
df_caracterizacion["CARGA_HOGAR"].value_counts()

CARGA_HOGAR
HOGAR_SIN_NINOS    466
CON_NINOS_1_2      267
SOLO_SIN_NINOS      81
CON_NINOS_3_MAS     13
DESCONOCIDO          3
Name: count, dtype: int64

In [37]:
df_caracterizacion["DISCAPACIDAD_AUTISTA"].value_counts()

DISCAPACIDAD_AUTISTA
Ninguna                                                 770
Autismo                                                   1
Tratorno generalizado del desarrollo no especificado      1
Sindrome de Asperger                                      1
Name: count, dtype: int64

In [38]:
df_caracterizacion["DISCAPACIDAD_INTELECTUAL"].value_counts()

DISCAPACIDAD_INTELECTUAL
Si    429
No    344
Name: count, dtype: int64

In [39]:
df_caracterizacion["DISCAPACIDAD_FISICA"].value_counts()

DISCAPACIDAD_FISICA
Ninguna    773
Name: count, dtype: int64

In [40]:
df_caracterizacion["DISCAPACIDAD_MENTAL_PSICOSOCIAL"].value_counts()

DISCAPACIDAD_MENTAL_PSICOSOCIAL
Ninguna                       754
Trastorno de Ansiedad           8
Trastorno afectivo bipolar      6
Otra                            3
Trastorno depresivo mayor       1
Esquizofrenia                   1
Name: count, dtype: int64

In [41]:
aut = df_caracterizacion["DISCAPACIDAD_AUTISTA"].astype(str).str.strip().str.upper().ne("NINGUNA")
inte = df_caracterizacion["DISCAPACIDAD_INTELECTUAL"].astype(str).str.strip().str.upper().isin(["SI","SÍ","TRUE"])
fis = df_caracterizacion["DISCAPACIDAD_FISICA"].astype(str).str.strip().str.upper().ne("NINGUNA")
psi = df_caracterizacion["DISCAPACIDAD_MENTAL_PSICOSOCIAL"].astype(str).str.strip().str.upper().ne("NINGUNA")

n = aut.astype(int) + inte.astype(int) + fis.astype(int) + psi.astype(int)

cond = np.where(n==0, "SIN",
         np.where(n>1, "MULTIPLE",
           np.where(inte | psi, "COGNITIVA", "OTRA")))

df_caracterizacion["CONDICION_DISCAPACIDAD"] = pd.Categorical(
    cond, categories=["SIN","COGNITIVA","OTRA","MULTIPLE"], ordered=True
)

In [42]:
df_caracterizacion["TRATAMIENTO_MEDICO"].value_counts()

TRATAMIENTO_MEDICO
No    801
Si     26
Name: count, dtype: int64

In [43]:
df_caracterizacion["ESTA_EMBARAZO"].value_counts()

ESTA_EMBARAZO
No    825
Si      2
Name: count, dtype: int64

Se sugiere eliminar embarazo por la frecuencia que tiene no hay mucha aleatoriedad

In [44]:
df_caracterizacion["FUMA"].value_counts()

FUMA
Nunca                        763
Una vez al mes                16
4 o más veces a la semana      9
2 a 4 veces al mes             6
2 o 3 veces a la semana        5
Name: count, dtype: int64

In [45]:
df_caracterizacion["CONSUME_ALCOHOL"].value_counts()

CONSUME_ALCOHOL
Nunca                      566
Una vez al mes             187
2 o 4 veces al mes          42
2 o 3 veces a la semana      4
Name: count, dtype: int64

In [46]:
risk = {
    "Nunca": 0, "Una vez al mes": 1,
    "2 a 4 veces al mes": 2,
    "2 o 3 veces a la semana": 3, "4 o más veces a la semana": 3
}

r_f = df_caracterizacion["FUMA"].map(risk).fillna(0)
r_a = df_caracterizacion["CONSUME_ALCOHOL"].map(risk).fillna(0)

df_caracterizacion["HABITOS_RIESGO"] = (
    pd.Series(np.maximum(r_f, r_a), index=df_caracterizacion.index)
      .map({0:"NINGUNO",1:"BAJO",2:"MODERADO",3:"ALTO"})
      .astype("category")
)

In [47]:
df_caracterizacion["HABITOS_RIESGO"].value_counts()

HABITOS_RIESGO
NINGUNO     622
BAJO        185
ALTO         18
MODERADO      5
Name: count, dtype: int64

HÁBITOS_RIESGO sintetiza en una sola escala (NINGUNO/BAJO/MODERADO/ALTO) la exposición conductual por consumo de alcohol y cigarrillo, reduciendo ruido y sparsidad frente a usar ambas variables por separado.

In [48]:
df_caracterizacion["PESO"].value_counts()

PESO
60      71
65      42
70      34
64      33
55      31
        ..
6465     1
67kg     1
617      1
66kg     1
421      1
Name: count, Length: 149, dtype: int64

In [49]:
df_caracterizacion["ESTATURA"].value_counts()

ESTATURA
1.70           59
1.75           41
1.80           34
1.60           29
1.78           28
               ..
189             1
1.761.76        1
1.47            1
1.86 metros     1
1z52            1
Name: count, Length: 132, dtype: int64

In [50]:
sp = (df_caracterizacion["PESO"].astype(str).str.lower().str.strip()
        .str.replace(",", ".", regex=False))
sp = sp.str.extract(r'([0-9]+(?:\.[0-9]+)?)', expand=False)         # toma el primer número
peso = pd.to_numeric(sp, errors="coerce")

# arregla cosas como 617 -> 61.7 ; 6643 -> 66.43
peso = peso.mask((peso >= 300) & (peso < 1000),    peso/10)
peso = peso.mask((peso >= 1000) & (peso < 100000), peso/100)

# rango plausible (kg)
peso = peso.where((peso >= 30) & (peso <= 250))
df_caracterizacion["PESO"] = peso.round(1)

# --- ESTATURA -> metros ---
se = (df_caracterizacion["ESTATURA"].astype(str).str.lower().str.strip()
        .str.replace(",", ".", regex=False))
se = se.str.extract(r'([0-9]+(?:\.[0-9]+)?)', expand=False)
est = pd.to_numeric(se, errors="coerce")

# si parece centímetros (>=100) pásalo a metros
est = est.mask(est >= 100, est/100)

# rango plausible (m)
est = est.where((est >= 1.30) & (est <= 2.20))
df_caracterizacion["ESTATURA"] = est.round(2)

In [51]:
imc = pd.to_numeric(df_caracterizacion["PESO"], errors="coerce") / (
      pd.to_numeric(df_caracterizacion["ESTATURA"], errors="coerce") ** 2)

df_caracterizacion["IMC"] = imc.round(1)

df_caracterizacion["IMC_CAT"] = pd.cut(
    df_caracterizacion["IMC"],
    bins=[-np.inf, 18.5, 25, 30, np.inf],
    labels=["BAJO_PESO", "NORMAL", "SOBREPESO", "OBESIDAD"]
).astype("category")

In [52]:
df_caracterizacion["IMC_CAT"].value_counts()


IMC_CAT
NORMAL       592
SOBREPESO    111
BAJO_PESO     97
OBESIDAD      21
Name: count, dtype: int64

In [53]:
OTROS_ESTUDIOS

NameError: name 'OTROS_ESTUDIOS' is not defined

In [ ]:
cols_drop = [
    "HORARIO_INICIO_TRANSPORTE", "HORARIO_FIN_TRANSPORTE", "BARRIO_PUNTO_PARTIDA",
    "ESTRATO_VIVIENDA_RESIDENCIA", "ESTRATO_VIVIENDA_RESIDENCIA_ORIGEN",
    "CUENTA_COMPUTADOR", "CUENTA_SMARTHPHONE_TABLET", "CUENTA_INTERNET",
    "GRUPO", "SUBGRUPO", "TIPO_USUARIO_EPS", "NUM_HERMANOS", "POSICION_HERMANOS",
    "TRABAJA_ACTUALMENTE", "HORAS_TRABAJADAS", "VIVE_PERSONAS_AREA_SALUD",
    "PERSONAS_VIVE", "NUM_NIÑOS_VIVE", "AYUDA_DESPLAZAMIENTO",
    "SORDOSEGUERA", "BARRERAS_UNIVERSIDAD", "INSCRITO_RULCPD", "TEMAS_INTERES",
    "ACTIVIDADES_PARTICIPA_UNIVERSIDAD", "ACTIVIDADES_TIEMPO_LIBRE",
    "SOLICITUDES_ASOCIADAS", "PRESENTA_ENFERMEDAD", "ENFERMEDADES_CRONICAS",
    "DISCAPACIDAD_AUTISTA", "DISCAPACIDAD_INTELECTUAL", "DISCAPACIDAD_FISICA",
    "SORDOCEGUERA", "DISCAPACIDAD_MENTAL_PSICOSOCIAL", "TRATAMIENTO_MEDICO",
    "FUMA", "CONSUME_ALCOHOL", "ESTA_EMBARAZO", "CONSUME_SUSTANCIAS_PSICOACTIVAS",
    "TIENE_TITULO", "OTROS_ESTUDIOS", "IMC", "PESO", "ESTATURA"
]




In [ ]:
# Mantén solo las que existan realmente en el DF y elimina
to_drop = [c for c in cols_drop if c in df_caracterizacion.columns]
df_caracterizacion.drop(columns=to_drop, inplace=True, errors="ignore")

print(f"Eliminadas {len(to_drop)} columnas:")
print(to_drop)

Eliminadas 42 columnas:
['HORARIO_INICIO_TRANSPORTE', 'HORARIO_FIN_TRANSPORTE', 'BARRIO_PUNTO_PARTIDA', 'ESTRATO_VIVIENDA_RESIDENCIA', 'ESTRATO_VIVIENDA_RESIDENCIA_ORIGEN', 'CUENTA_COMPUTADOR', 'CUENTA_SMARTHPHONE_TABLET', 'CUENTA_INTERNET', 'GRUPO', 'SUBGRUPO', 'TIPO_USUARIO_EPS', 'NUM_HERMANOS', 'POSICION_HERMANOS', 'TRABAJA_ACTUALMENTE', 'HORAS_TRABAJADAS', 'VIVE_PERSONAS_AREA_SALUD', 'PERSONAS_VIVE', 'NUM_NIÑOS_VIVE', 'AYUDA_DESPLAZAMIENTO', 'SORDOSEGUERA', 'BARRERAS_UNIVERSIDAD', 'INSCRITO_RULCPD', 'TEMAS_INTERES', 'ACTIVIDADES_PARTICIPA_UNIVERSIDAD', 'ACTIVIDADES_TIEMPO_LIBRE', 'SOLICITUDES_ASOCIADAS', 'PRESENTA_ENFERMEDAD', 'ENFERMEDADES_CRONICAS', 'DISCAPACIDAD_AUTISTA', 'DISCAPACIDAD_INTELECTUAL', 'DISCAPACIDAD_FISICA', 'DISCAPACIDAD_MENTAL_PSICOSOCIAL', 'TRATAMIENTO_MEDICO', 'FUMA', 'CONSUME_ALCOHOL', 'ESTA_EMBARAZO', 'CONSUME_SUSTANCIAS_PSICOACTIVAS', 'TIENE_TITULO', 'OTROS_ESTUDIOS', 'IMC', 'PESO', 'ESTATURA']


In [ ]:
df_caracterizacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830 entries, 0 to 829
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   DOCUMENTO                        830 non-null    int64   
 1   GENERO                           830 non-null    object  
 2   UBICACION_SEMESTRAL              830 non-null    int64   
 3   MEDIO_TRANSPORTE                 830 non-null    object  
 4   FRECUENCIA_TRANSPORTE            830 non-null    object  
 5   FUENTES_FINANCIACION             830 non-null    object  
 6   PERSONAS_A_CARGO                 830 non-null    category
 7   DIFICULTAD_PAGO_MATRICULA        830 non-null    object  
 8   TURNO_TRABAJO_ESTUDIO            830 non-null    object  
 9   TIPO_VIVIENDA_RESIDENCIA         830 non-null    object  
 10  ZONA_VIVIENDA_RESIDENCIA         830 non-null    object  
 11  VIVE_PADRES_FAMILIAR             830 non-null    object  
 12  SISBEN  

In [ ]:
df_caracterizacion.to_csv("../data/df_caracterizacion_recategorizacion_variables_depurada.csv", index=False)